<a href="https://colab.research.google.com/github/faeze-bnt/Attack-project/blob/colab/Attack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install larq

In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import SGD

import tensorflow as tf

import os

# Enable XLA devices
tf.config.optimizer.set_jit(True)

In [3]:
import tensorflow as tf
import larq as lq

In [4]:
from tensorflow.keras.models import Sequential

In [5]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))

# Normalize pixel values to be between -1 and 1
train_images, test_images = train_images / 127.5 - 1, test_images / 127.5 - 1

In [6]:
def train_bnn(data, file_name_n, params, num_epochs=50, batch_size=128, train_temp=1, init=None):

    #params = [32, 32, 64, 64, 200, 200]

    kwargs = dict(input_quantizer="ste_sign",
              kernel_quantizer="ste_sign",
              kernel_constraint="weight_clip")

    model = Sequential()

    # In the first layer we only quantize the weights and not the input
    model.add(lq.layers.QuantConv2D(params[0], (3, 3),
                                kernel_quantizer="ste_sign",
                                kernel_constraint="weight_clip",
                                #use_bias=False,
                                input_shape=(28, 28, 1)))#data.train_data.shape[1:]))

    model.add(Activation('relu'))
    model.add(lq.layers.QuantConv2D(params[1], (3, 3), **kwargs))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(lq.layers.QuantConv2D(params[2], (3, 3), **kwargs))
    model.add(Activation('relu'))
    model.add(lq.layers.QuantConv2D(params[3], (3, 3), **kwargs))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())
    model.add(lq.layers.QuantDense(params[4], **kwargs))
    model.add(Activation('relu'))
    model.add(Dropout(0.5))
    model.add(lq.layers.QuantDense(params[5], **kwargs))
    model.add(Activation('relu'))
    model.add(lq.layers.QuantDense(10, **kwargs))


    lq.models.summary(model)

    return model

In [7]:
train()


+sequential stats------------------------------------------------------------------------------------+
| Layer            Input prec.           Outputs  # 1-bit  # 32-bit  Memory  1-bit MACs  32-bit MACs |
|                        (bit)                        x 1       x 1    (kB)                          |
+----------------------------------------------------------------------------------------------------+
| quant_conv2d               -  (-1, 26, 26, 32)      288        32    0.16           0       194688 |
| activation                 -  (-1, 26, 26, 32)        0         0       0           ?            ? |
| quant_conv2d_1             1  (-1, 24, 24, 32)     9216        32    1.25     5308416            0 |
| activation_1               -  (-1, 24, 24, 32)        0         0       0           ?            ? |
| max_pooling2d              -  (-1, 12, 12, 32)        0         0       0           0            0 |
| quant_conv2d_2             1  (-1, 10, 10, 64)    18432        64    2.